<a href="https://colab.research.google.com/github/ddesiccantt/EECS498-DC/blob/main/A1_Q2_kNN_prediction_6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
# batch processing
import torch

train_points = torch.tensor([
    [0., 0.],
    [1., 0.],
    [0., 1.],
    [5., 5.],
    [6., 5.]
])

train_labels = torch.tensor([0, 0, 0, 1, 1])

queries = torch.tensor([
    [1., 1.],
    [5., 4.],
    [6., 6.],
    [0., 0.],
    [5., 5.]
])

# for batch in queries.split(2):
#     print(batch)

def knn_predict_fast(train_points, train_labels, queries, k=3):
    query_squared = (queries ** 2).sum(dim=1, keepdim=True)
    train_squared = (train_points ** 2).sum(dim=1).unsqueeze(0)
    dot_products = queries @ train_points.T

    squared_distances = (
        query_squared + train_squared - 2 * dot_products
    )

    # 浮点计算可能产生极小的负数，将它们设为 0
    squared_distances = squared_distances.clamp(min=0)

    _, indices = squared_distances.topk(
        k=k, dim=1, largest=False
    )
    neighbor_labels = train_labels[indices]

    results = []
    for labels in neighbor_labels:
        prediction = torch.bincount(labels).argmax()
        results.append(prediction)

    return torch.stack(results)

# results = []

# for batch in queries.split(2):
#     predictions = knn_predict_fast(
#         train_points, train_labels, batch, k=3
#     )
#     results.append(predictions)

# print(results)
# predictions = torch.cat(results, dim=0)

# print(predictions)
# tensor([0, 1, 1, 0, 1])
def knn_predict_batched(
    train_points, train_labels, queries, k=3, batch_size=2
):
    results = []

    for batch in queries.split(batch_size):
        predictions = knn_predict_fast(
            train_points, train_labels, batch, k=k
        )
        results.append(predictions)

    return torch.cat(results, dim=0)

queries = torch.tensor([
    [1., 1.],
    [5., 4.],
    [6., 6.],
    [0., 0.],
    [5., 5.]
])

predictions = knn_predict_batched(
    train_points, train_labels, queries,
    k=3, batch_size=3
)

print(predictions)
# tensor([0, 1, 1, 0, 1])

tensor([0, 1, 1, 0, 1])


In [2]:
import torch

train_points = torch.tensor([
    [0., 0.],
    [1., 0.],
    [0., 1.],
    [5., 5.],
    [6., 5.]
])

train_labels = torch.tensor([0, 0, 0, 1, 1])

torch.manual_seed(42)
order = torch.randperm(train_points.shape[0])

shuffled_points = train_points[order]
shuffled_labels = train_labels[order]

tensor([2, 4, 3, 0, 1])


In [5]:
import torch

def knn_predict_fast(train_points, train_labels, queries, k=3):
    query_squared = (queries ** 2).sum(dim=1, keepdim=True)
    train_squared = (train_points ** 2).sum(dim=1).unsqueeze(0)
    dot_products = queries @ train_points.T

    squared_distances = (
        query_squared + train_squared - 2 * dot_products
    )

    # 浮点计算可能产生极小的负数，将它们设为 0
    squared_distances = squared_distances.clamp(min=0)

    _, indices = squared_distances.topk(
        k=k, dim=1, largest=False
    )
    neighbor_labels = train_labels[indices]

    results = []
    for labels in neighbor_labels:
        prediction = torch.bincount(labels).argmax()
        results.append(prediction)

    return torch.stack(results)

points = torch.tensor([
    [0., 0.],
    [1., 0.],
    [0., 1.],
    [1., 1.],
    [5., 5.],
    [6., 5.],
    [5., 6.],
    [6., 6.]
])
labels = torch.tensor([0, 0, 0, 0, 1, 1, 1, 1])

torch.manual_seed(42)
order = torch.randperm(points.shape[0])

# 前 6 个索引用于训练，剩余 2 个用于验证
train_indices = order[:6]
val_indices = order[6:]

train_points = points[train_indices]
train_labels = labels[train_indices]

val_points = points[val_indices]
val_labels = labels[val_indices]

best_k = None
best_accuracy = -1

for k in [1, 3, 5]:
    predictions = knn_predict_fast(
        train_points, train_labels, val_points, k=k
    )

    accuracy = (predictions == val_labels).float().mean().item()

    print(f"k={k}, 验证准确率={accuracy:.2%}")

    if accuracy >= best_accuracy:
        best_accuracy = accuracy
        best_k = k

print("选出的 k：", best_k)

k=1, 验证准确率=100.00%
k=3, 验证准确率=100.00%
k=5, 验证准确率=0.00%
选出的 k： 3
